# 01 - Bronze Demo: Taxi Trips

Notebook này giúp kiểm tra kết quả của PART B - Task 1: Bronze Layer Ingestion.

Mục tiêu kiểm tra:

- Đọc raw taxi trip JSON bằng PySpark.
- Xem các bản ghi có malformed dates, missing coordinates/location IDs và duplicate records.
- Đọc Delta Bronze table và kiểm tra metadata ingestion.
- Thử append một batch mới theo đúng cơ chế append-only.

Trong project này, đường dẫn /data/bronze/taxi_trips/ được lưu dưới dạng data/bronze/taxi_trips/ tương đối với thư mục gốc project để chạy được trên Windows.


In [ ]:
from pathlib import Path
import sys

from pyspark.sql import functions as F


def find_project_root(start: Path) -> Path:
    """Find the project directory from the current notebook working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bronze" / "bronze_ingestion.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bronze.bronze_ingestion import RAW_SCHEMA, create_spark, ingest_batch

RAW_DIR = PROJECT_ROOT / "data" / "raw_json"
BATCH_1 = RAW_DIR / "batch_001.json"
BATCH_2 = RAW_DIR / "batch_002.json"
BRONZE_PATH = PROJECT_ROOT / "data" / "bronze" / "taxi_trips"

for required_path in (BATCH_1, BATCH_2, BRONZE_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required path: {required_path}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw batch 1:  {BATCH_1}")
print(f"Raw batch 2:  {BATCH_2}")
print(f"Bronze path: {BRONZE_PATH}")


In [ ]:
spark = create_spark()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")


## 1. Inspect raw JSON input

Bronze giữ nguyên các giá trị raw. Vì vậy, date được đọc dưới dạng string để malformed dates không làm mất bản ghi trong bước ingestion.


In [ ]:
raw_batch_1 = spark.read.schema(RAW_SCHEMA).json(str(BATCH_1))
raw_batch_2 = spark.read.schema(RAW_SCHEMA).json(str(BATCH_2))
raw_all = raw_batch_1.unionByName(raw_batch_2, allowMissingColumns=True).cache()

print(f"Batch 1 rows: {raw_batch_1.count():,}")
print(f"Batch 2 rows: {raw_batch_2.count():,}")
print(f"Raw rows total: {raw_all.count():,}")
raw_all.printSchema()

raw_all.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "pickup_latitude",
    "pickup_longitude",
    "dropoff_latitude",
    "dropoff_longitude",
).show(10, truncate=False)


## 2. Check dirty records in the raw input

Các chỉ số dưới đây chỉ để quan sát Bronze input; chưa làm sạch hay loại bỏ bản ghi.


In [ ]:
pickup_ts = F.to_timestamp("tpep_pickup_datetime")
dropoff_ts = F.to_timestamp("tpep_dropoff_datetime")

quality_summary = raw_all.select(
    F.count("*").alias("raw_rows"),
    F.sum(
        F.when(F.col("tpep_pickup_datetime").isNotNull() & pickup_ts.isNull(), 1).otherwise(0)
    ).alias("malformed_pickup_dates"),
    F.sum(
        F.when(F.col("tpep_dropoff_datetime").isNotNull() & dropoff_ts.isNull(), 1).otherwise(0)
    ).alias("malformed_dropoff_dates"),
    F.sum(
        F.when(
            F.col("PULocationID").isNull() | F.col("DOLocationID").isNull(),
            1,
        ).otherwise(0)
    ).alias("rows_with_missing_location_ids"),
    F.sum(
        F.when(
            F.col("pickup_latitude").isNull()
            | F.col("pickup_longitude").isNull()
            | F.col("dropoff_latitude").isNull()
            | F.col("dropoff_longitude").isNull(),
            1,
        ).otherwise(0)
    ).alias("rows_with_missing_coordinates"),
).collect()[0]

duplicate_groups = (
    raw_all.groupBy("trip_id")
    .count()
    .where(F.col("count") > 1)
)
duplicate_group_count = duplicate_groups.count()
duplicate_row_count = (
    duplicate_groups.select(F.sum("count").alias("duplicate_rows")).collect()[0][0]
    or 0
)

print("Raw quality summary:")
for field in quality_summary.__fields__:
    print(f"- {field}: {quality_summary[field]:,}")
print(f"- duplicate trip_id groups: {duplicate_group_count:,}")
print(f"- rows belonging to duplicate groups: {duplicate_row_count:,}")

print()
print("Examples of dirty records:")
raw_all.where(
    (pickup_ts.isNull() & F.col("tpep_pickup_datetime").isNotNull())
    | F.col("PULocationID").isNull()
    | F.col("pickup_latitude").isNull()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "pickup_latitude",
    "pickup_longitude",
).show(10, truncate=False)


## 3. Read the existing Bronze Delta table

Bronze table có thêm metadata phục vụ audit: source_file, ingest_batch_id, ingested_at và raw_record_hash.


In [ ]:
bronze_before = (
    spark.read.format("delta").load(str(BRONZE_PATH)).cache()
)

print(f"Bronze rows: {bronze_before.count():,}")
bronze_before.printSchema()

print("Rows by ingest batch:")
bronze_before.groupBy("ingest_batch_id").count().orderBy("ingest_batch_id").show(truncate=False)

print("Bronze sample:")
bronze_before.select(
    "trip_id",
    "tpep_pickup_datetime",
    "PULocationID",
    "pickup_latitude",
    "source_file",
    "ingest_batch_id",
    "ingested_at",
    "raw_record_hash",
).show(10, truncate=False)


## 4. Test append-only ingestion with a new batch

Cell này append batch_002.json với ingest_batch_id riêng cho demo. Nếu chạy lại notebook, cell sẽ bỏ qua batch đã tồn tại để tránh append trùng ngoài ý muốn.


In [ ]:
APPEND_TEST_BATCH_ID = "demo_append_batch_002"

existing_batch_ids = {
    row["ingest_batch_id"]
    for row in bronze_before.select("ingest_batch_id").distinct().collect()
}

before_append_count = bronze_before.count()
if APPEND_TEST_BATCH_ID in existing_batch_ids:
    print(f"Skip: {APPEND_TEST_BATCH_ID!r} already exists.")
else:
    appended_rows = ingest_batch(
        spark=spark,
        input_path=BATCH_2,
        output_path=BRONZE_PATH,
        batch_id=APPEND_TEST_BATCH_ID,
    )
    print(f"Appended rows: {appended_rows:,}")

bronze_after = spark.read.format("delta").load(str(BRONZE_PATH)).cache()
after_append_count = bronze_after.count()
print(f"Bronze rows before append: {before_append_count:,}")
print(f"Bronze rows after append:  {after_append_count:,}")
print(f"Row-count increase:        {after_append_count - before_append_count:,}")

bronze_after.groupBy("ingest_batch_id").count().orderBy("ingest_batch_id").show(truncate=False)


## 5. Final Bronze checks

Các check cuối giúp xác nhận dữ liệu lỗi vẫn được giữ lại trong Bronze và duplicate chưa bị deduplicate ở layer này. Việc làm sạch/deduplicate sẽ thuộc Silver layer.


In [ ]:
bronze_after.select(
    F.count("*").alias("bronze_rows"),
    F.countDistinct("raw_record_hash").alias("distinct_raw_hashes"),
    F.countDistinct("ingest_batch_id").alias("ingest_batches"),
).show()

print("Duplicate raw hashes in Bronze:")
bronze_after.groupBy("raw_record_hash").count().where(F.col("count") > 1).show(10, truncate=False)

print("Malformed pickup-date examples retained in Bronze:")
bronze_after.where(
    F.col("tpep_pickup_datetime").isNotNull()
    & F.to_timestamp("tpep_pickup_datetime").isNull()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "ingest_batch_id",
).show(10, truncate=False)


In [ ]:
# Chạy cell này khi kết thúc notebook để giải phóng Spark session.
spark.stop()
